In [32]:
import pandas as pd
df = pd.read_csv("Bank_Dataset.csv")
df


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,01-01-2024,3.11,AMAZON SELLER SVCS,Debit,₹2462,678275,UPI,TXN190872
1,01-Jan-24,5.44,BHIM-BMTC,DR,50,681007,UPI,TXN143064
2,01-Jan-24,9.35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728,NEFT,TXN246316
3,01-01-2024,14.07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745,UPI,TXN569226
4,01-Jan-24,14.23,BHIM-BLINKIT,Debit,270,680737,UPI,TXN968962
...,...,...,...,...,...,...,...,...
1323,30-06-2024,11.53,SWIGGY-INSTAMART,DR,637,-311783,UPI,TXN274261
1324,30-Jun-24,16.35,BHIM ZEPTO,DR,693,-312476,UPI,TXN792601
1325,30-06-2024,17.12,UPI-STARBUCKS@AXIS,DR,327,-310557,UPI,TXN859233
1326,30-06-2024,19.56,POS ZOMATO,DR,₹589,-311146,UPI,TXN492729


**Feature 1:** the transaction parser

In [19]:
#record inital row count
total_initial_rows = len(df)

# Drop the 18 exact duplicate rows
df = df.drop_duplicates()
dropped_duplicates = total_initial_rows - len(df)

# Parse and Clean Date Format (Handles all 4 formats dynamically)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
unparseable_dates = df['Date'].isna().sum()

#Parse and Clean Amount Format
df['Amount'] = df['Amount'].astype(str)
# Chain clean replacements for currency notations, prefixes, and formatting marks
df['Amount'] = (df['Amount']
                .str.replace('₹', '', regex=False)
                .str.replace('â‚¹', '', regex=False) # Safely handles corrupted encoding strings if present
                .str.replace('Rs.', '', regex=False)
                .str.replace(',', '', regex=False)
                .str.strip())
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
unparseable_amounts = df['Amount'].isna().sum()

# 5. Standardise the Type field to lowercase 'debit' and 'credit'
type_mapping = {
    'DR': 'debit', 'Debit': 'debit', 'debit': 'debit',
    'CR': 'credit', 'Credit': 'credit', 'credit': 'credit'
}
df['Type'] = df['Type'].astype(str).str.strip().map(type_mapping)

# 6. Treat empty or whitespace strings in 'Mode' as missing (NaN)
if 'Mode' in df.columns:
    df['Mode'] = df['Mode'].astype(str).str.strip().replace('', np.nan)

# 7. Print the exact requested summary format string
total_parsed = len(df)
print(f"Parsed {total_parsed} transactions across 6 months. "
      f"Dropped {dropped_duplicates} duplicates. "
      f"{unparseable_amounts} unparseable amounts, "
      f"{unparseable_dates} unparseable dates.")

Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 567 unparseable dates.


feature 2: Vendor extractor


In [20]:
#vendor keyword distionary
vendor_keywords = {
    'Swiggy': ['SWIGGY', 'BUNDL', 'INSTAMART'],
    'Zomato': ['ZOMATO', 'EMPIRE RESTAURANT', 'MEGHANA FOODS', 'TRUFFLES', 'DINEOUT'],
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'Zepto': ['ZEPTO'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Uber': ['UBER'],
    'Ola': ['OLA CABS', 'OLA-PRIME', 'ANI TECHNOLOGIES', 'OLA ELECTRIC'],
    'Rapido': ['RAPIDO', 'ROPPEN'],
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS', 'TWC', 'THIRD WAVE', 'THIRDWAVE'],
    'Cafe Coffee Day': ['CCD', 'COFFEE DAY'],
    'Dmart': ['DMART', 'AVENUE SUPERMARTS'],
    'BigBasket': ['BIGBASKET'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Hotstar': ['HOTSTAR', 'DISNEY'],
    'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW'],
    'Salary Credit': ['SALARY', 'TECHCRUSH'],
    'Rent Expense': ['RENT', 'LANDLORD'],
    'Electricity Bill': ['BESCOM', 'BANGALORE ELEC'],
    'Water Bill': ['BWSSB'],
    'Telecom / Jio / Airtel': ['JIO', 'AIRTEL', 'VI POSTPAID'],
    'Fuel Station': ['HP PETROL', 'BPCL', 'INDIAN OIL', 'PETROL PUMP'],
    'BMTC Bus': ['BMTC', 'TUMMOC'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS MOVIE'],
    'Nykaa': ['NYKAA']
}

#defind exact extraction logic
def extract_vendor(description):
  if not isinstance(description, str):
    return 'Uncategorized'
  desc_upper = description.upper()

  #1.check for ATM withdrawls
  if 'ATM-WDL' in desc_upper or 'CASH-WDL' in desc_upper:
    return 'Cash withdrawl'

  #2.identitfy p2p transfers before checking cormrical merchants
  p2p_handles = ['@OK', '@HDFC', '@PAYTM', '@AXIS', '@ICICI']
  all_keywords = []
  for keywords in vendor_keywords.values():
    all_keywords.extend(keywords)

  if not any(kw in desc_upper for kw in all_keywords):
    return 'p2p_Transfer'

  #3. check canonical names keywords
  for vendor, keywords in vendor_keywords.items():
    for keyword in keywords:
      if keyword in desc_upper:
        return vendor
  return 'Uncategorised'

#apply to descitpion column to create vendor_clean
df['vendor_clean'] = df['Description'].apply(extract_vendor)

#print
print(df['vendor_clean'].nunique())
print("\n-----top 10 most frequent vendors -----")

print(df['vendor_clean'].value_counts().head(10))

30

-----top 10 most frequent vendors -----
vendor_clean
Swiggy          243
Zomato          165
p2p_Transfer     97
Amazon           86
Ola              77
Starbucks        73
Uber             71
Zepto            58
Blinkit          55
Rapido           55
Name: count, dtype: int64


Feature 3: Category Tagger

In [21]:
# 1. Define the Vendor-to-Category Mapping Dictionary
category_mapping = {
    # Food Delivery & Restaurants & Cafes
    'Swiggy': 'Food Delivery',
    'Zomato': 'Restaurants',
    'Starbucks': 'Cafe',
    'Cafe Coffee Day': 'Cafe',
    'Third Wave Coffee': 'Cafe',

    # Quick Commerce & Groceries
    'Blinkit': 'Quick Commerce',
    'Zepto': 'Quick Commerce',
    'Dmart': 'Groceries',
    'BigBasket': 'Groceries',

    # E-commerce
    'Amazon': 'Ecommerce',
    'Flipkart': 'Ecommerce',
    'Myntra': 'Ecommerce',
    'Nykaa': 'Ecommerce',

    # Transport
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Rapido': 'Transport',
    'BMTC Bus': 'Transport',

    # Fixed Bills, Utilities & Subscriptions
    'Electricity Bill': 'Utilities',
    'Water Bill': 'Utilities',
    'Telecom / Jio / Airtel': 'Utilities',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Hotstar': 'Subscriptions',
    'Rent Expense': 'Rent Expense',

    # Investments, Fuel & Entertainment
    'Zerodha': 'Investments',
    'Groww': 'Investments',
    'Fuel Station': 'Fuel',
    'BookMyShow': 'Entertainment',
    'Salary Credit': 'Salary',

    # Special Non-Merchant Structural Mappings
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal'
}

#map vendor clean to the new category columns
df['category'] = df['vendor_clean'].map(category_mapping)

#handle any uncategorised item safely to avoid missing from the platform
df['category'] = df['category'].fillna('Uncategorised')

#Expected Analytical Outcome verification
print("--- Transaction Count by Category ---")
print(df['category'].value_counts())


--- Transaction Count by Category ---
category
Food Delivery     243
Transport         240
Ecommerce         169
Restaurants       165
Uncategorised     114
Quick Commerce    113
Cafe               99
Utilities          38
Groceries          33
Subscriptions      28
Investments        23
Fuel               22
Entertainment      11
Salary              6
Rent Expense        6
Name: count, dtype: int64


feature 4: Spending overview


In [23]:
total_transaction_count = len(df)

total_debits = df[df['Type'] == 'debit']['Amount'].sum()
total_credits = df[df['Type'] == 'credit']['Amount'].sum()

net_savings = total_credits - total_debits

savings_rate = (net_savings / total_credits) * 100 if total_credits != 0 else 0

#extract top 5 breakdown data
debit_df = df[df['Type']=='debit']

top_categories = debit_df.groupby('category')['Amount'].sum().sort_values(ascending = False).head(5)
top_vendors = debit_df.groupby('vendor_clean')['Amount'].sum().sort_values(ascending = False).head(5)

print("=========================================")
print("       FEATURE 4: SPENDING OVERVIEW      ")
print("=========================================")
print(f"Total Transaction Count : {total_transaction_count}")
print(f"Total Credits           : ₹{total_credits:,.2f}")
print(f"Total Debits            : ₹{total_debits:,.2f}")
print(f"Net Savings             : ₹{net_savings:,.2f}")
print(f"Savings Rate            : {savings_rate:.2f}%")
print("-----------------------------------------")

print("\nTop 5 Catergories by spend")
for rank, (cat, amt) in enumerate(top_categories.items(), 1):
  print(f"{rank}. {cat:<20}: ₹{amt:,.2f}")

print("\nTOP 5 VENDORS BY SPEND:")
for rank, (vendor, amt) in enumerate(top_vendors.items(), 1):
    print(f"{rank}. {vendor:<20} : ₹{amt:,.2f}")
print("=========================================")

       FEATURE 4: SPENDING OVERVIEW      
Total Transaction Count : 1310
Total Credits           : ₹509,774.00
Total Debits            : ₹1,678,901.00
Net Savings             : ₹-1,169,127.00
Savings Rate            : -229.34%
-----------------------------------------

Top 5 Catergories by spend
1. Ecommerce           : ₹599,581.00
2. Investments         : ₹248,160.00
3. Uncategorised       : ₹158,563.00
4. Restaurants         : ₹128,075.00
5. Rent Expense        : ₹108,000.00

TOP 5 VENDORS BY SPEND:
1. Amazon               : ₹328,530.00
2. Zerodha              : ₹210,000.00
3. Flipkart             : ₹177,510.00
4. Zomato               : ₹128,075.00
5. p2p_Transfer         : ₹113,063.00


features 5: monthly trend analysis

In [28]:
df['month_num'] = df['Date'].dt.month

#month name instead of numbers
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
df['month'] = df['month_num'].map(month_names)

# Redefine debit_df to include the newly created 'month' column
debit_df = df[df['Type']=='debit']

#build matrix using table
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
month_pivot = debit_df.pivot_table(
    values = 'Amount',
    index = 'category',
    columns = 'month',
    aggfunc = 'sum'
).reindex(columns = month_order).fillna(0)

#percentage growth / decline btw 1 and last month
jan_spend = month_pivot['Jan']
june_spend = month_pivot['Jun']
growth_series = ((june_spend - jan_spend) / jan_spend * 100).replace([float('inf'), -float('inf')], 0)

biggest_growth_cat = growth_series.idxmax()
biggest_growth_val = growth_series.max()

biggest_decline_cat = growth_series.idxmin()
biggest_decline_val = growth_series.min()

print("=========================================================================")
print("                    FEATURE 5: MONTHLY TREND ANALYSIS                   ")
print("=========================================================================")
print(month_pivot.round(2))
print("-------------------------------------------------------------------------")
print(f"Trending Up   : {biggest_growth_cat} shot up by {biggest_growth_val:.1f}% from Jan to Jun.")
print(f"Trending Down : {biggest_decline_cat} dropped by {biggest_decline_val:.1f}% from Jan to Jun.")
print("=========================================================================")

                    FEATURE 5: MONTHLY TREND ANALYSIS                   
month               Jan      Feb      Mar      Apr      May      Jun
category                                                            
Cafe             2162.0   1932.0   3742.0   3616.0   1853.0   4245.0
Ecommerce       54786.0  66141.0  36780.0  41209.0  57670.0  25705.0
Entertainment     952.0      0.0   2418.0   1178.0      0.0    897.0
Food Delivery    7496.0  12331.0  11639.0  11325.0  10033.0   9760.0
Fuel             8901.0   1452.0  13669.0   9663.0   3328.0   2882.0
Groceries        4740.0   1517.0   4295.0   4453.0   4444.0   3715.0
Investments     18956.0  15000.0  48761.0  34737.0  48628.0  19496.0
Quick Commerce   5672.0   5763.0   4372.0   4729.0   6372.0   3859.0
Rent Expense    18000.0      0.0  18000.0  18000.0  18000.0  18000.0
Restaurants      9425.0  11579.0  24571.0  11307.0  18010.0  14078.0
Subscriptions    1176.0   2711.0   3071.0    488.0    913.0   3436.0
Transport        7119.0   4527

Feature 6 - Time-of-Day Patterns

In [31]:
import numpy as np
#extract hours component from time coulmn
df['hour'] = df['Time'].astype(str).str.split('.').str[0].astype(int)

#filter credit/salary to look at outbound lifestyle
debit_df = df[df['Type']  == 'debit']

#build pivot matrix
hour_columns = list(range(24))
time_matrix_df = debit_df.pivot_table(
    values = 'Amount',
    index = 'category',
    columns = 'hour',
    aggfunc = 'sum'
).reindex(columns = hour_columns).fillna(0).astype(int)

#pandas dataframe ->numpy array
time_matrix_np = time_matrix_df.to_numpy()
categories = time_matrix_df.index.tolist()

#calculate lifestyle behaviors
#swiggy orders btw 9pm to 1 am
swiggy_total = debit_df[debit_df['vendor_clean'] == 'Swiggy'].shape[0]
swiggy_late_night = debit_df[(debit_df['vendor_clean'] == 'Swiggy') & (debit_df['hour'].isin([21,22,23,0]))].shape[0]
swiggy_percentage = (swiggy_late_night / swiggy_total) * 100 if swiggy_total != 0 else 0

cafe_total = debit_df[debit_df['category'] == 'Cafe'].shape[0]
cafe_morning = debit_df[(debit_df['category'] == 'Cafe') & (debit_df['hour'].isin([8, 9, 10]))].shape[0]
cafe_percentage = (cafe_morning / cafe_total * 100) if cafe_total > 0 else 0

print("==========================================================================================")
print("                          FEATURE 6: TIME-OF-DAY PATTERNS                                 ")
print("==========================================================================================")
print("PRINTABLE DISTRIBUTION BARS (Total activity density across 24 hours):")
print("------------------------------------------------------------------------------------------")

# CORRECTED LOOP: Notice the print() is aligned with the inner 'for' statement, not inside it!
for idx, cat in enumerate(categories):
    row_data = time_matrix_np[idx]
    total_row_activity = row_data.sum()

    # Complete the full 24-hour bar string first
    visual_bar = ""
    for count in row_data:
        if count == 0:
            visual_bar += "░" # Zero activity
        elif count <= 2:
            visual_bar += "▒" # Light activity
        else:
            visual_bar += "▓" # Dense activity peak

    # UN-INDENTED: Prints exactly one solid final timeline bar per category
    print(f"{cat:<20} |{visual_bar}| (Total: {total_row_activity} txns)")

print("------------------------------------------------------------------------------------------")
print("------------------------------------------------------------------------------------------")
print(f"Late-Night Craving Alert : {swiggy_percentage:.1f}% of Swiggy orders happen between 9 PM and 1 AM.")
print(f"Morning Fueling Habits   : {cafe_percentage:.1f}% of Cafe transactions happen during commute hours (8-11 AM).")
print("==========================================================================================\n")

#heatmap
print("VISUAL HEATMAP MATRIX GRID")
time_matrix_df.style.background_gradient(cmap='YlOrRd', axis=None).format(precision=0)

                          FEATURE 6: TIME-OF-DAY PATTERNS                                 
PRINTABLE DISTRIBUTION BARS (Total activity density across 24 hours):
------------------------------------------------------------------------------------------
Cafe                 |▓▓░▓░░▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░▓| (Total: 31445 txns)
Ecommerce            |▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓| (Total: 599581 txns)
Entertainment        |▓▓░░░░░▓▓░░░░░░░▓░▓▓▓░▓░| (Total: 6773 txns)
Food Delivery        |▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓| (Total: 104351 txns)
Fuel                 |▓░░▓▓▓░░▓▓░▓▓▓▓▓▓░░▓░▓░▓| (Total: 74255 txns)
Groceries            |▓▓▓▓▓░▓░▓▓▓▓▓▓▓░░░▓▓▓▓░▓| (Total: 50208 txns)
Investments          |▓▓░░▓▓▓░░▓▓░▓░░░░░▓▓▓░▓▓| (Total: 248160 txns)
Quick Commerce       |▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓| (Total: 60152 txns)
Rent Expense         |░░░░░░░░░░░▓░▓▓░░░░░░░░░| (Total: 108000 txns)
Restaurants          |▓▓▓▓▓▓░▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓| (Total: 128075 txns)
Subscriptions        |░▓▓▓▓▓▓▓░░░▓▓▓▓░░░░▓░▓░░| (Total: 15619 tx

hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
category,,,,,,,,,,,,,,,,,,,,,,,,
Cafe,872,549,0,316,0,0,179,480,2668,1986,4227,2102,1163,1454,1332,2085,3079,3552,2562,584,1491,0,0,764
Ecommerce,18650,9016,10246,14865,12018,17258,9249,10916,6537,40724,15613,54468,24116,23660,72584,6427,52046,35378,35272,12668,51190,35095,16182,15403
Entertainment,562,620,0,0,0,0,0,311,1463,0,0,0,0,0,0,0,835,0,1270,417,952,0,343,0
Food Delivery,1077,2313,1056,1213,1860,2383,238,931,1857,4536,2229,6301,8284,5106,3508,5116,3512,3003,9221,9525,11687,8755,5678,4962
Fuel,2675,0,0,676,2010,2105,0,0,1727,2283,0,2598,9159,8350,7828,11937,8334,0,0,1087,0,11845,0,1641
Groceries,3895,1536,2352,7080,1169,0,5265,0,2430,4684,3524,4940,502,1228,655,0,0,0,910,1248,2566,3688,0,2536
Investments,4883,15000,0,0,18834,4496,37717,0,0,4737,90000,0,4389,0,0,0,0,0,15000,4476,15000,0,15000,18628
Quick Commerce,859,1402,697,732,768,1448,787,252,1625,5491,2063,181,3952,2622,2443,3014,3554,948,4355,6992,6574,5569,3061,763
Rent Expense,0,0,0,0,0,0,0,0,0,0,0,36000,0,54000,18000,0,0,0,0,0,0,0,0,0


Feature 7: Anomaly Detection


In [37]:
# Define vendor_keywords (copied from an earlier cell to ensure it's available)
vendor_keywords = {
    'Swiggy': ['SWIGGY', 'BUNDL', 'INSTAMART'],
    'Zomato': ['ZOMATO', 'EMPIRE RESTAURANT', 'MEGHANA FOODS', 'TRUFFLES', 'DINEOUT'],
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'Zepto': ['ZEPTO'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Uber': ['UBER'],
    'Ola': ['OLA CABS', 'OLA-PRIME', 'ANI TECHNOLOGIES', 'OLA ELECTRIC'],
    'Rapido': ['RAPIDO', 'ROPPEN'],
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS', 'TWC', 'THIRD WAVE', 'THIRDWAVE'],
    'Cafe Coffee Day': ['CCD', 'COFFEE DAY'],
    'Dmart': ['DMART', 'AVENUE SUPERMARTS'],
    'BigBasket': ['BIGBASKET'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Hotstar': ['HOTSTAR', 'DISNEY'],
    'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW'],
    'Salary Credit': ['SALARY', 'TECHCRUSH'],
    'Rent Expense': ['RENT', 'LANDLORD'],
    'Electricity Bill': ['BESCOM', 'BANGALORE ELEC'],
    'Water Bill': ['BWSSB'],
    'Telecom / Jio / Airtel': ['JIO', 'AIRTEL', 'VI POSTPAID'],
    'Fuel Station': ['HP PETROL', 'BPCL', 'INDIAN OIL', 'PETROL PUMP'],
    'BMTC Bus': ['BMTC', 'TUMMOC'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS MOVIE'],
    'Nykaa': ['NYKAA']
}

# Define extract_vendor function (copied from an earlier cell)
def extract_vendor(description):
  if not isinstance(description, str):
    return 'Uncategorised'
  desc_upper = description.upper()

  #1.check for ATM withdrawls
  if 'ATM-WDL' in desc_upper or 'CASH-WDL' in desc_upper:
    return 'Cash Withdrawal'

  #2.identitfy p2p transfers before checking cormrical merchants
  p2p_handles = ['@OK', '@HDFC', '@PAYTM', '@AXIS', '@ICICI']
  all_keywords = []
  for keywords in vendor_keywords.values():
    all_keywords.extend(keywords)

  if not any(kw in desc_upper for kw in all_keywords):
    return 'P2P Transfer'

  #3. check canonical names keywords
  for vendor, keywords in vendor_keywords.items():
    for keyword in keywords:
      if keyword in desc_upper:
        return vendor
  return 'Uncategorised'

# Apply to description column to create vendor_clean
df['vendor_clean'] = df['Description'].apply(extract_vendor)

# Define category_mapping (copied from an earlier cell)
category_mapping = {
    # Food Delivery & Restaurants & Cafes
    'Swiggy': 'Food Delivery',
    'Zomato': 'Restaurants',
    'Starbucks': 'Cafe',
    'Cafe Coffee Day': 'Cafe',
    'Third Wave Coffee': 'Cafe',

    # Quick Commerce & Groceries
    'Blinkit': 'Quick Commerce',
    'Zepto': 'Quick Commerce',
    'Dmart': 'Groceries',
    'BigBasket': 'Groceries',

    # E-commerce
    'Amazon': 'Ecommerce',
    'Flipkart': 'Ecommerce',
    'Myntra': 'Ecommerce',
    'Nykaa': 'Ecommerce',

    # Transport
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Rapido': 'Transport',
    'BMTC Bus': 'Transport',

    # Fixed Bills, Utilities & Subscriptions
    'Electricity Bill': 'Utilities',
    'Water Bill': 'Utilities',
    'Telecom / Jio / Airtel': 'Utilities',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Hotstar': 'Subscriptions',
    'Rent Expense': 'Rent Expense',

    # Investments, Fuel & Entertainment
    'Zerodha': 'Investments',
    'Groww': 'Investments',
    'Fuel Station': 'Fuel',
    'BookMyShow': 'Entertainment',
    'Salary Credit': 'Salary',

    # Special Non-Merchant Structural Mappings
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal'
}

# Map vendor_clean to the new category columns
df['category'] = df['vendor_clean'].map(category_mapping)

# Handle any uncategorized items safely to avoid missing from the platform
df['category'] = df['category'].fillna('Uncategorised')

debit_df = df[df['Type'] == 'debit'].copy()

#compute grp specific mean and SD
debit_df['mean_amount'] = debit_df.groupby('category')['Amount'].transform('mean')
debit_df['std_amount'] = debit_df.groupby("category")['Amount'].transform('std')

#Calculate the exact Z-Score for every individual debit transaction
# Formula: Z = (X - μ) / σ
debit_df['z_score'] = (debit_df['Amount'] -debit_df['mean_amount']) / (debit_df['std_amount'] + 1e-5)

#apply stadard statistical treshold freq
anomalies_df = debit_df[debit_df['z_score'] > 2].copy()

#sort by most severe deviation
anamolies_sorted = anomalies_df.sort_values(by = 'z_score',ascending = False)

#Verified section Summary & High-Value Insights
print("==========================================================================================")
print("                             FEATURE 7: ANOMALY DETECTION                                 ")
print("=========================================================================================")
print(f"Total Lifestyle Anomalies Flagged (Z-Score > 2): {len(anamolies_sorted)} transactions")
print("------------------------------------------------------------------------------------------")
print("TOP 5 MOST SEVERE FINANCIAL ANOMALIES:")
print("------------------------------------------------------------------------------------------")


top_5_anomalies = anamolies_sorted[['Date', 'vendor_clean', 'category', 'Amount', 'z_score']].head(5)


#cleary aligned table
print(f"{'Date':<12} | {'Vendor':<18} | {'Category':<18} | {'Amount': <10} | {'Z-Score': <8}")

print("-"*75)

for idx , row in top_5_anomalies.iterrows():
  date_str = row['Date'].strftime('%d-%b-%Y') if hasattr(row['Date'], 'strftime') else str(row['Date'])
  print(f"{date_str:<12} | {row['vendor_clean']:<18} | {row['category']:<18} | ₹{row['Amount']:<9,.2f} | {row['z_score']:.2f}")

print("="*75)

                             FEATURE 7: ANOMALY DETECTION                                 
Total Lifestyle Anomalies Flagged (Z-Score > 2): 0 transactions
------------------------------------------------------------------------------------------
TOP 5 MOST SEVERE FINANCIAL ANOMALIES:
------------------------------------------------------------------------------------------
Date         | Vendor             | Category           | Amount     | Z-Score 
---------------------------------------------------------------------------


feature 8: Spending Archetype Detection


In [44]:
import numpy as np
#ensure we have a clean copy of outbound expenses for behavioral matching
# Ensure 'hour' column is available for archetype detection within this cell
if 'hour' not in df.columns:
    df['hour'] = df['Time'].astype(str).str.split('.').str[0].astype(int)

debit_df = df[df['Type'] == 'debit'].copy() # Corrected filtering for 'debit' transactions
total_debit_spend = debit_df['Amount'].sum()

def check_foodie(df, total_spend):
  food_categories = ['food delivery', 'Restaurants', 'Cafe'] # Corrected 'Resturants' to 'Restaurants'
  food_spend = df[df['category'].isin(food_categories)]['Amount'].sum()
  percentage = (food_spend / total_spend * 100) if total_spend > 0 else 0 # Corrected syntax
  return percentage > 35, f"{percentage:.1f}% of total budget spent on dining/cafes"

def check_investor(df):
  investment_count = df[df['category'] == 'Investments'].shape[0]
  return investment_count >= 3, f"Executed {investment_count} investment transaction"

def check_late_night_snacker(df):
  target_vendors =['Swiggy', 'Zepto', 'Blinkit']
  vendor_txns = df[df['vendor_clean'].isin(target_vendors)]
  total_orders = vendor_txns.shape[0]

  late_night_orders = vendor_txns[vendor_txns['hour'].isin([21, 22, 23, 0])].shape[0]
  percentage = (late_night_orders / total_orders * 100) if total_orders > 0 else 0 # Corrected syntax
  return percentage > 20, f"{percentage:.1f}% of quick supply orders happen late night"

def check_compulsive_shopper(df):
    # Rule: More than 15 transactions in Ecommerce (Amazon, Flipkart, etc.)
    ecommerce_count = df[df['category'] == 'Ecommerce'].shape[0]
    return ecommerce_count > 15, f"Logged {ecommerce_count} e-commerce purchases"

# 2. Run the detection engine to map matched personalities
matched_archetypes = {}

# Evaluate all rules independently
is_foodie, foodie_metric = check_foodie(debit_df, total_debit_spend)
if is_foodie: matched_archetypes['The Foodie'] = foodie_metric

is_investor, investor_metric = check_investor(debit_df)
if is_investor: matched_archetypes['The Investor'] = investor_metric

is_snacker, snacker_metric = check_late_night_snacker(debit_df)
if is_snacker: matched_archetypes['The Late-Night Snacker'] = snacker_metric

is_shopper, shopper_metric = check_compulsive_shopper(debit_df)
if is_shopper: matched_archetypes['The Compulsive Shopper'] = shopper_metric


print("                          FEATURE 8: SPENDING ARCHETYPE DETECTION                        ")
print(f"Analyzing behavioral fingerprints for Rahul...")


if matched_archetypes:
    print("MATCHED FINANCIAL ARCHETYPES:")
    print("-" * 55)
    for archetype, metric in matched_archetypes.items():
        print(f"{archetype:<28} -> Supporting Evidence: {metric}")
else:
    print("Neutral Spender: No extreme behavioral archetypes triggered.")

                          FEATURE 8: SPENDING ARCHETYPE DETECTION                        
Analyzing behavioral fingerprints for Rahul...
Neutral Spender: No extreme behavioral archetypes triggered.


Final report

In [50]:
# =========================================================================
# FEATURE 9 : FINAL SPENDDNA REPORT
# =========================================================================
# Ensure 'Date' column is in datetime format
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

# Ensure 'Amount' column is numeric
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# Standardise the Type field to lowercase 'debit' and 'credit' for robustness
type_mapping = {
    'DR': 'debit', 'Debit': 'debit', 'debit': 'debit',
    'CR': 'credit', 'Credit': 'credit', 'credit': 'credit'
}
df['Type'] = df['Type'].astype(str).str.strip().map(type_mapping)

# Ensure vendor_clean and category are available, re-run if needed based on context
# Assuming extract_vendor and category_mapping functions/dictionaries are defined globally or in previous cells
# If not, they would need to be copied here for self-containment.

# Re-create debit_df to ensure it's up-to-date and populated
debit_df = df[df['Type'] == 'debit'].copy()

# Recalculate key metrics to ensure they are based on the current df state
total_credits = df[df['Type'] == 'credit']['Amount'].sum()
total_debits = debit_df['Amount'].sum()
net_savings = total_credits - total_debits
savings_rate = (net_savings / total_credits) * 100 if total_credits != 0 else 0


print("\n")
print("="*75)
print("SpendDNA REPORT".center(75))
print("="*75)

start_date = df['Date'].min().strftime("%b %Y")
end_date = df['Date'].max().strftime("%b %Y")
print("SpendDNA REPORT - RAHUL SHARMA")
print(f"{len(df):,} months | {len(df):,} transactions | {start_date} to {end_date}")
print("="*75)

# -------------------------------------------------------------------------
# EXECUTIVE SUMMARY
# -------------------------------------------------------------------------
print("\nEXECUTIVE SUMMARY")

print(f"Total Credits      : Rs. {total_credits:,.0f}")
print(f"Total Debits       : Rs. {total_debits:,.0f}")

status = "RUNNING SAVINGS" if net_savings >= 0 else "OVERSPENDING"

print(f"Net Change         : Rs. {net_savings:,.0f}")
print(f"Savings Rate       : {savings_rate:.1f}% ({status})")
print(f"Transactions       : {len(df):,}")
print(f"Unique Vendors     : {df['vendor_clean'].nunique()}")

# -------------------------------------------------------------------------
# TOP CATEGORIES
# -------------------------------------------------------------------------
print("\nTOP CATEGORIES (% of debit total)")

if total_debits > 0:
    category_spend = debit_df.groupby("category")["Amount"].sum().sort_values(ascending=False)

    for cat, amt in category_spend.head(5).items():

        pct = amt / total_debits * 100

        bar = "█"*int(pct/2)

        print(f"{cat:<20} {pct:5.1f}% {bar:<20} Rs. {amt:,.0f}")
else:
    print("No debit transactions to analyze categories.")

# -------------------------------------------------------------------------
# TOP VENDORS
# -------------------------------------------------------------------------
print("\nTOP VENDORS")

if not debit_df.empty:
    vendor_amount = debit_df.groupby("vendor_clean")["Amount"].sum().sort_values(ascending=False)
    vendor_count = debit_df["vendor_clean"].value_counts()

    for vendor in vendor_amount.head(5).index:

        amt = vendor_amount[vendor]
        cnt = vendor_count[vendor]

        print(f"{vendor:<18} Rs. {amt:>10,.0f} ({cnt} orders)")
else:
    print("No debit transactions to analyze vendors.")

# -------------------------------------------------------------------------
# TIME OF DAY
# -------------------------------------------------------------------------
print("\nTIME-OF-DAY PATTERNS")

if "Time" in debit_df.columns and not debit_df.empty:

    debit_df["Hour"] = pd.to_numeric(debit_df["Time"], errors="coerce")

    night = debit_df[(debit_df["Hour"]>=21)|(debit_df["Hour"]<=1)]
    morning = debit_df[(debit_df["Hour"]>=6)&(debit_df["Hour"]<=11)]

    # Add check for len(debit_df) to prevent ZeroDivisionError
    if len(debit_df) > 0:
        night_pct = len(night)/len(debit_df)*100
        morning_pct = len(morning)/len(debit_df)*100

        print(f"Food Delivery Peak : 21:00 - 01:00 ({night_pct:.0f}% of orders)")
        print(f"Cafe Peak          : 09:00 - 11:00 ({morning_pct:.0f}% morning runs)")
        print("Quick Commerce     : evenly distributed")
    else:
        print("No debit transactions for time-of-day analysis.")
else:
    print("No 'Time' column or no debit transactions for time-of-day analysis.")

# -------------------------------------------------------------------------
# MONTHLY TREND
# -------------------------------------------------------------------------
print("\nMONTHLY TREND")

if not debit_df.empty:
    # Ensure 'month' column is created before grouping
    if 'month_num' not in df.columns:
        df['month_num'] = df['Date'].dt.month
    if 'month' not in df.columns:
        month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
        df['month'] = df['month_num'].map(month_names)
    # Re-create debit_df to include the 'month' column if it was just added to df
    debit_df = df[df['Type'] == 'debit'].copy()

    month_order = ['Jan','Feb','Mar','Apr','May','Jun']
    monthly = debit_df.groupby("month")["Amount"].sum().reindex(
        month_order
    )

    mx = monthly.max()

    for month, amt in monthly.items():

        if pd.isna(amt) or mx == 0:
            bar = "░"*20 # Represent as empty bar if no data or max is zero
            print(f"{month:<4} Rs. {0:>8,.0f} {bar}") # Print 0 amount for NA months
        else:
            bar = "█"*int((amt/mx)*20)
            print(f"{month:<4} Rs. {amt:>8,.0f} {bar}")
else:
    print("No debit transactions for monthly trend analysis.")

# -------------------------------------------------------------------------
# TOP ANOMALIES
# -------------------------------------------------------------------------
print("\nTOP ANOMALIES (3+ stddev from category mean)")

# Placeholder for `extreme_spends`. Assuming anamolies_sorted is from Feature 7.
# If `anamolies_sorted` is empty or not defined, provide a default.
if 'anamolies_sorted' not in globals() or anamolies_sorted.empty:
    extreme_spends = pd.DataFrame(columns=['Date', 'vendor_clean', 'category', 'Amount', 'z_score'])
else:
    extreme_spends = anamolies_sorted

if len(extreme_spends)>0 and not debit_df.empty:
    # Ensure mean_spend and std_spend are calculated from the current debit_df
    mean_spend = debit_df['Amount'].mean()
    std_spend = debit_df['Amount'].std()

    print(f"{'Date':<12} | {'Vendor':<18} | {'Category':<18} | {'Amount': <10} | {'Z-Score': <8}")
    print("-"*75)

    for _,row in extreme_spends.sort_values("Amount",ascending=False).head(5).iterrows():
        # Handle potential division by zero if std_spend is 0 for a category
        if std_spend != 0:
            z = (row["Amount"]-mean_spend)/std_spend
        else:
            z = 0 # Or handle as appropriate, e.g., 'N/A'

        date_str = row['Date'].strftime('%d %b') if pd.notna(row['Date']) and hasattr(row['Date'], 'strftime') else str(row['Date'])
        print(f"{date_str:<12} | {row['vendor_clean']:<18} | {row['category']:<18} | ₹{row['Amount']:<9,.0f} | {z:.1f}σ")

else:
    print("No significant anomalies detected.")

# -------------------------------------------------------------------------
# SPENDING ARCHETYPES
# -------------------------------------------------------------------------
print("\nRAHUL'S SPENDING ARCHETYPES")

if total_debits > 0:
    food_pct = category_spend.get("Food Delivery",0)/total_debits*100
    quick_pct = category_spend.get("Quick Commerce",0)/total_debits*100
    shop_pct = category_spend.get("Ecommerce",0)/total_debits*100
    invest_pct = category_spend.get("Investments",0)/total_debits*100

    is_snacker_local = False
    if not debit_df.empty and 'hour' in debit_df.columns:
        target_vendors =['Swiggy', 'Zepto', 'Blinkit']
        vendor_txns = debit_df[debit_df['vendor_clean'].isin(target_vendors)]
        total_orders = vendor_txns.shape[0]
        if total_orders > 0:
            late_night_orders = vendor_txns[vendor_txns['hour'].isin([21, 22, 23, 0])].shape[0]
            percentage = (late_night_orders / total_orders * 100)
            if percentage > 20:
                is_snacker_local = True

    if food_pct>10:
        print(f"-> THE FOODIE              ({food_pct:.1f}% on food)")

    if quick_pct>10:
        print(f"-> THE QUICK COMMERCE      ({quick_pct:.1f}% on q-commerce)")

    if shop_pct>10:
        print(f"-> THE SHOPAHOLIC          ({shop_pct:.1f}% on e-commerce)")

    if invest_pct>5:
        print(f"-> THE INVESTOR            ({invest_pct:.1f}% on investments)")

    if is_snacker_local:
        print("-> THE LATE-NIGHT SNACKER")

    if savings_rate<0:
        print("-> THE YOLO SPENDER")
    else:
        print("-> THE SMART SAVER")
else:
    print("No debit transactions for archetype detection.")

# -------------------------------------------------------------------------
# KEY INSIGHTS
# -------------------------------------------------------------------------
print("\nKEY INSIGHTS")

salary = df[df["category"]=="Salary"]["Amount"].sum()

if salary>0:

    monthly_salary = salary/6

    print(f"1. Rahul is earning around Rs. {monthly_salary:,.0f} per month.")

if savings_rate<0:
    print(f"2. Spending exceeds income by {-savings_rate:.1f}%.")

if not debit_df.empty:
    largest_cat = category_spend.idxmax()
    print(f"3. Highest spending category : {largest_cat}")

    largest_vendor = vendor_amount.idxmax()
    print(f"4. Favourite vendor : {largest_vendor}")
else:
    print("No debit transactions for key insights on categories/vendors.")

if invest_pct>0:
    print(f"5. Investments contribute {invest_pct:.1f}% of total debit spending.")

print("="*75)
print("END OF REPORT".center(75))
print("="*75)



                              SpendDNA REPORT                              
SpendDNA REPORT - RAHUL SHARMA
1,328 months | 1,328 transactions | Jan 2024 to Jun 2024

EXECUTIVE SUMMARY
Total Credits      : Rs. 85,393
Total Debits       : Rs. 505,487
Net Change         : Rs. -420,094
Savings Rate       : -492.0% (OVERSPENDING)
Transactions       : 1,328
Unique Vendors     : 30

TOP CATEGORIES (% of debit total)
Ecommerce             40.0% ███████████████████  Rs. 202,017
Investments           15.6% ███████              Rs. 78,956
Food Delivery          9.0% ████                 Rs. 45,570
Restaurants            6.6% ███                  Rs. 33,463
Quick Commerce         6.3% ███                  Rs. 31,812

TOP VENDORS
Flipkart           Rs.     94,526 (48 orders)
Amazon             Rs.     87,716 (87 orders)
Zerodha            Rs.     75,000 (14 orders)
Swiggy             Rs.     45,570 (248 orders)
Zomato             Rs.     33,463 (168 orders)

TIME-OF-DAY PATTERNS
Food Delivery Peak